# ETL de los Partidos de Fútbol

Este *notebook* accede a la API semiprivada de la LaLiga para extraer el calendario de eventos futbolísticos para las principales competiciones. En caso de que la API esté caducada, devolverá un dataframe vacío y un aviso.

In [1]:
# Importar librerías
import time as time_module
import pandas as pd
from datetime import datetime, date, timedelta, time

In [2]:
def fetch_all_competitions_matches(start_date="2024-01-01", end_date="2025-05-25", delay=0.5):
    """
    Descarga partidos de fútbol desde la API de LaLiga para múltiples competiciones,
    entre start_date y end_date (ambas incluidas). El "subscriptionSlug" se construye a partir
    del nombre base de la competición más el año (2023, 2024 o 2025) según corresponda.

    Args:
        - start_date (str): Fecha inicial en formato "YYYY-MM-DD".
        - end_date   (str): Fecha final en formato "YYYY-MM-DD".
        - delay      (float): Tiempo de espera entre peticiones, en segundos.

    Returns:
        - DataFrame con columnas: fecha, hora, local, visitante, competición, temporada, jornada.
          Si no se obtienen datos, devuelve un DataFrame vacío y muestra un aviso.
    """
    base_url = "https://apim.laliga.com/public-service/api/v1/matches"
    headers = {"User-Agent": "Mozilla/5.0"}

    # Lista de competiciones base (sin año)
    competitions = [
        "laliga-easports",
        "copa-del-rey",
        "primera-division-femenina",
        "supercopa-de-espana",
        "laliga-hypermotion",
        "copa-de-la-reina",
        "champions-league",
        "europa-conference-league",
        "europa-league",
        "uefa-eurocopa",
        "supercopa-de-europa"
    ]

    # Años posibles en los slugs (pueden empezar en 2023, 2024 o 2025 según competición)
    years = ["2023", "2024", "2025"]

    # Convertir cadenas a objetos datetime.date para filtrar luego
    start_dt = datetime.fromisoformat(start_date).date()
    end_dt   = datetime.fromisoformat(end_date).date()

    all_matches = []

    # Generar combinaciones de competición + año
    for comp in competitions:
        for yr in years:
            slug = f"{comp}-{yr}"
            
            # Para cada combinación de slug, iteramos por semanas (1..60 como máximo aproximado)
            for week in range(1, 61):
                params = {
                    "subscriptionSlug": slug,
                    "week": week,
                    "limit": 100,
                    "orderField": "date",
                    "orderType": "asc",
                    "contentLanguage": "es",
                    "countryCode": "GB",
                    "subscription-key": "c13c3a8e2f6b46da9c5c425cf61fab3e" # API-key
                }
                try:
                    res = requests.get(base_url, headers=headers, params=params)
                    res.raise_for_status()
                    data = res.json()
                    matches = data.get("matches", [])

                    # Si no hay partidos, asumimos que ya superamos las jornadas de esa temporada
                    if not matches:
                        break

                    for m in matches:
                        try:
                            # Fecha completa en formato ISO: "2023-08-11T17:30:00+00:00"
                            datetime_obj = datetime.fromisoformat(m["date"].replace("Z", "+00:00"))
                            match_date = datetime_obj.date()
                            match_time = datetime_obj.time()

                            # Solo agregar si está dentro del rango deseado
                            if start_dt <= match_date <= end_dt:
                                competicion_name = m.get("competition", {}).get("name", "Desconocido")
                                local = m.get("home_team", {}).get("shortname", "Desconocido")
                                visitante = m.get("away_team", {}).get("shortname", "Desconocido")
                                jornada = m.get("gameweek", {}).get("week", None)

                                all_matches.append({
                                    "fecha": match_date,
                                    "hora": match_time,
                                    "local": local,
                                    "visitante": visitante,
                                    "competicion": competicion_name,
                                    "temporada_slug": slug,
                                    "jornada": jornada
                                })
                        except Exception:
                            # Ignorar partido si hay error al parsear
                            continue

                    time_module.sleep(delay)

                except Exception:
                    # Si la petición falla (slug inválido o semana fuera de rango), salimos del bucle de semanas
                    break

    # Construir DataFrame y ordenarlo solo si hay datos
    df_all = pd.DataFrame(all_matches)

    if df_all.empty:
        print("No se ha podido acceder a los datos a través de la API proporcionada")
        return df_all  # Devuelve vacío, evita errores posteriores

    df_all = df_all.sort_values(["fecha", "hora"]).reset_index(drop=True)
    df_all["temporada_slug"].replace("-", "_", regex=True, inplace=True)
    
    return df_all

In [3]:
def normalise_df_text(df, rename_vars=True, rename_values=True):
    """
    Normaliza el texto de un DataFrame aplicando transformaciones estándar:
    minúsculas, snake_case (espacios -> "_") y sin acentos.

    Args:
        - df (pd.DataFrame): DataFrame a normalizar
        - rename_vars (bool): Si True, normaliza nombres de columnas
        - rename_values (bool): Si True, normaliza valores de texto (solo columnas object)

    Returns:
        - pd.DataFrame: DataFrame normalizado
    """
    # Usar translate
    accent_translation = str.maketrans({
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u",
    })

    df_normalised = df.copy()

    # Valores (solo columnas tipo object)
    if rename_values:
        text_cols = df_normalised.select_dtypes(include=["object"]).columns
        df_normalised[text_cols] = (
            df_normalised[text_cols]
            .apply(lambda s: (
                s.astype("string")                      # manejar NaN con seguridad
                 .str.strip()                           # eliminar espacios antes y después
                 .str.lower()                           # convertir a minúsculas
                 .str.translate(accent_translation)     # quitar acentos carácter a carácter
                 .str.replace(r"\s+", "_", regex=True)  # espacios en blanco -> "_"
            ))
        )

    # Nombres de columnas
    if rename_vars:
        df_normalised.columns = [
            str(c).strip().lower().translate(accent_translation).replace(" ", "_")
            for c in df_normalised.columns
        ]

    return df_normalised

In [4]:
# Obtener partidos
timer_start = time_module.time()
df_all_football_competitions = (
    fetch_all_competitions_matches("2024-01-01", "2025-05-25", delay=0.05)
    .pipe(normalise_df_text)
)
timer_end = time_module.time()

timer_seconds = timer_end - timer_start
print(f"Duración de las peticiones API de los partidos de fútbol:"
      f"\n{timer_seconds:.2f} s"
      f"\n{timer_seconds / 60:.2f} m"
      f"\n{timer_seconds / 3600:.2f} h")

In [5]:
if not df_all_football_competitions.empty:
    # Generar variable "datetime" y eliminar "fecha" y "hora"
    df_all_football_competitions["fecha"] = pd.to_datetime(df_all_football_competitions["fecha"]).dt.date
    df_all_football_competitions["hora"] = pd.to_datetime(df_all_football_competitions["hora"], format="%H:%M:%S").dt.time
    df_all_football_competitions["datetime"] = df_all_football_competitions.apply(
        lambda row: datetime.combine(row["fecha"], row["hora"]),
        axis=1
    )
    df_all_football_competitions.drop(columns=["fecha", "hora"], inplace=True)
    
    # Guardar .csv
    store_path = "C:\\Users\\David\\Desktop\\Master\\Modulo TFM\\Notebooks modulares\\raw data"
    df_all_football_competitions.to_csv(os.path.join(store_path, "all_football_competitions.csv"), index=False)